# 过滤器

过滤器允许您根据提供的条件从结果集中包含或排除特定对象。     
有关过滤器运算符的列表，请参阅API 参考页面。



In [9]:
import json
import weaviate
from weaviate.auth import AuthApiKey

# 连接到本地部署的 Weaviate
client = weaviate.Client(
    url="http://127.0.0.1:8080",
    auth_client_secret=AuthApiKey("WVF5YThaHlkYwhGUSmCRgsX3tD5ngdN8pkih")
)

## 使用一个条件进行过滤
向您的查询添加一个filter，以限制结果集。

过滤器结构

过滤where器是一个代数对象，它采用以下参数：

Operator（采用下列值之一）
- And
- Or
- Equal
- NotEqual
- GreaterThan
- GreaterThanEqual
- LessThan
- LessThanEqual
- Like
- WithinGeoRange
- IsNull
- ContainsAny （*仅适用于数组和文本属性）
- ContainsAll （*仅适用于数组和文本属性）
- Path：是XPath样式的字符串列表，表示集合的属性名称。
如果属性是交叉引用，则路径后面应跟一个字符串列表。对于引用集合inPublication的引用属性Publication，其路径选择器name应为["inPublication", "Publication", "name"]。
- valueType
- valueInt：针对int数据类型。
- valueBoolean：针对boolean数据类型。
- valueString：用于string数据类型（注意：string已弃用）。
- valueText：适用于text、uuid、geoCoordinates、phoneNumber数据类型。
- valueNumber：针对number数据类型。
- valueDate：对于date（ISO 8601 时间戳，格式为RFC3339）数据类型。
如果运算符是And或Or，则操作数是过滤器列表where。

In [10]:
response = (
    client.query
    .get("JeopardyQuestion", ["question", "answer"])
    .with_where({
        "path": ["answer"],
        "operator": "Like",
        "valueText": "We*"
    })
    .with_limit(3)
    .do()
)

print(json.dumps(response, indent=2))

{
  "data": {
    "Get": {
      "JeopardyQuestion": [
        {
          "answer": "Weaviate",
          "question": "This vector DB is OSS and supports automatic property type inference on import"
        },
        {
          "answer": "Weaviate",
          "question": "This vector DB is OSS and supports automatic property type inference on import"
        }
      ]
    }
  }
}


In [11]:
from weaviate.classes.query import Filter

jeopardy = client.collections.get("JeopardyQuestion")
response = jeopardy.query.fetch_objects(
    filters=Filter.by_property("round").equal("Double Jeopardy!"),
    limit=3
)

for o in response.objects:
    print(o.properties)

ModuleNotFoundError: No module named 'weaviate.classes'

## 具有多个条件的筛选
要使用两个或多个条件进行过滤，请使用And或Or来定义条件之间的关系。

In [ ]:
response = (
    client.query
    .get("JeopardyQuestion", ["question", "answer", "round", "points"])
    .with_where({
        "operator": "And",
        "operands": [
            {
                "path": ["round"],
                "operator": "Equal",
                "valueText": "Double Jeopardy!",
            },
            {
                "path": ["points"],
                "operator": "LessThan",
                "valueInt": 600,
            },
        ]
    })
    .with_limit(3)
    .do()
)

print(json.dumps(response, indent=2))

### 版本
Python 客户端APIv4提供通过any_of、 或all_of以及使用&或|运算符进行过滤。


- 使用any_of或all_of通过提供的过滤器列表中的任意一个或全部进行过滤。
- 使用&或|通过提供的过滤器对进行过滤。


In [ ]:
## & or |

from weaviate.classes.query import Filter

jeopardy = client.collections.get("JeopardyQuestion")
response = jeopardy.query.fetch_objects(
    # Use & as AND
    #     | as OR
    filters=(
        Filter.by_property("round").equal("Double Jeopardy!") &
        Filter.by_property("points").less_than(600)
    ),
    limit=3
)

for o in response.objects:
    print(o.properties)

In [ ]:
## any_of
from weaviate.classes.query import Filter

jeopardy = client.collections.get("JeopardyQuestion")
response = jeopardy.query.fetch_objects(
    filters=(
        Filter.any_of([  # Combines the below with `|`
            Filter.by_property("points").greater_or_equal(700),
            Filter.by_property("points").less_than(500),
            Filter.by_property("round").equal("Double Jeopardy!"),
        ])
    ),
    limit=5
)

for o in response.objects:
    print(o.properties)

In [ ]:
## all_of

from weaviate.classes.query import Filter

jeopardy = client.collections.get("JeopardyQuestion")
response = jeopardy.query.fetch_objects(
    filters=(
        Filter.all_of([  # Combines the below with `&`
            Filter.by_property("points").greater_than(300),
            Filter.by_property("points").less_than(700),
            Filter.by_property("round").equal("Double Jeopardy!"),
        ])
    ),
    limit=5
)

for o in response.objects:
    print(o.properties)

## 嵌套过滤器
您可以对过滤器进行分组和嵌套。

In [ ]:
response = (
    client.query
    .get("JeopardyQuestion", ["question", "answer", "round", "points"])
    .with_where({
        "operator": "And",
        "operands": [
            {
                "path": ["answer"],
                "operator": "Like",
                "valueText": "*bird*",
            },
            {
                "operator": "Or",
                "operands": [
                    {
                        "path": ["points"],
                        "operator": "GreaterThan",
                        "valueInt": 700,
                    },
                    {
                        "path": ["points"],
                        "operator": "LessThan",
                        "valueInt": 300,
                    },
                ]
            }
        ]
    })
    .with_limit(3)
    .do()
)

print(json.dumps(response, indent=2))

In [ ]:
from weaviate.classes.query import Filter

jeopardy = client.collections.get("JeopardyQuestion")
response = jeopardy.query.fetch_objects(
    filters=Filter.by_property("answer").like("*bird*") &
            (Filter.by_property("points").greater_than(700) | Filter.by_property("points").less_than(300)),
    limit=3
)

for o in response.objects:
    print(o.properties)

## 组合过滤器和搜索运算符
nearXXX过滤器可与诸如、hybrid、 和 之类的搜索运算符一起使用bm25。

In [ ]:
response = (
    client.query
    .get("JeopardyQuestion", ["question", "answer", "round", "points"])
    .with_where({
        "path": ["points"],
        "operator": "GreaterThan",
        "valueInt": 200
    })
    .with_near_text({
        "concepts": ["fashion icons"]
    })
    .with_limit(3)
    .do()
)

print(json.dumps(response, indent=2))

In [ ]:
from weaviate.classes.query import Filter

jeopardy = client.collections.get("JeopardyQuestion")
response = jeopardy.query.near_text(
    query="fashion icons",
    filters=Filter.by_property("points").greater_than(200),
    limit=3
)

for o in response.objects:
    print(o.properties)

## ContainsAny筛选
该ContainsAny运算符作用于文本属性，并以值数组作为输入。它将匹配属性包含数组中任意（即一个或多个）值的对象。

In [ ]:
token_list = ["australia", "india"]

response = (
    client.query
    .get("JeopardyQuestion", ["question", "answer", "round", "points"])
    # Find objects where the `answer` property contains any of the strings in `token_list`
    .with_where({
        "path": ["answer"],
        "operator": "ContainsAny",
        "valueText": token_list
    })
    .with_limit(3)
    .do()
)

print(json.dumps(response, indent=2))

In [ ]:
from weaviate.classes.query import Filter

jeopardy = client.collections.get("JeopardyQuestion")

token_list = ["australia", "india"]
response = jeopardy.query.fetch_objects(
    # Find objects where the `answer` property contains any of the strings in `token_list`
    filters=Filter.by_property("answer").contains_any(token_list),
    limit=3
)

for o in response.objects:
    print(o.properties)

## ContainsAll筛选

该ContainsAll运算符作用于文本属性，并以值数组作为输入。它将匹配属性包含数组中所有值的对象。

In [ ]:
token_list = ["blue", "red"]

response = (
    client.query
    .get("JeopardyQuestion", ["question", "answer", "round", "points"])
    .with_where({
        "path": ["question"],
        "operator": "ContainsAll",
        "valueText": token_list
    })
    .with_limit(3)
    .do()
)

print(json.dumps(response, indent=2))

In [ ]:
from weaviate.classes.query import Filter

jeopardy = client.collections.get("JeopardyQuestion")

token_list = ["blue", "red"]

response = jeopardy.query.fetch_objects(
    # Find objects where the `question` property contains all of the strings in `token_list`
    filters=Filter.by_property("question").contains_all(token_list),
    limit=3
)

for o in response.objects:
    print(o.properties)

## ContainsAny并ContainsAll批量删除
如果要进行批量删除，请参[阅删除对象](https://weaviate.io/developers/weaviate/manage-data/delete#containsany--containsall)。

## 根据部分匹配过滤文本
如果对象属性是text、 或text之类的数据类型（例如对象 ID），则使用Like来过滤部分文本匹配。

In [ ]:
response = (
    client.query
    .get("JeopardyQuestion", ["question", "answer", "round"])
    .with_where({
        "path": ["answer"],
        "operator": "Like",
        "valueText": "*inter*"
    })
    .with_limit(3)
    .do()
)

print(json.dumps(response, indent=2))

In [ ]:
from weaviate.classes.query import Filter

jeopardy = client.collections.get("JeopardyQuestion")
response = jeopardy.query.fetch_objects(
    filters=Filter.by_property("answer").like("*inter*"),
    limit=3
)

for o in response.objects:
    print(o.properties)

## 使用交叉引用进行过滤
要根据交叉引用对象的属性进行过滤，请将集合名称添加到过滤器中。


In [ ]:
response = (
    client.query
    .get("JeopardyQuestion", ["question", "answer", "round", "hasCategory {... on JeopardyCategory { title } }"])
    .with_where({
        "path": ["hasCategory", "JeopardyCategory", "title"],
        "operator": "Like",
        "valueText": "*Sport*"
    })
    .with_limit(3)
    .do()
)

print(json.dumps(response, indent=2))

In [ ]:
from weaviate.classes.query import Filter, QueryReference

jeopardy = client.collections.get("JeopardyQuestion")
response = jeopardy.query.fetch_objects(
    filters=Filter.by_ref(link_on="hasCategory").by_property("title").like("*Sport*"),
    return_references=QueryReference(link_on="hasCategory", return_properties=["title"]),
    limit=3
)

for o in response.objects:
    print(o.properties)
    print(o.references["hasCategory"].objects[0].properties["title"])

## 按地理坐标

目前，地理坐标过滤仅限于距离源位置最近的 800 个结果，任何其他过滤条件和搜索参数都会进一步减少该结果的数量。

如果您计划使用人口密集的数据集，请考虑使用另一种策略，例如将地理散列到text数据类型中，然后进一步过滤，例如使用ContainsAny过滤器。

In [ ]:
get_publications_where = """
  {
    Get {
      Publication(where: {
        operator: WithinGeoRange,
        valueGeoRange: {
          geoCoordinates: {
            latitude: 52.3932696,    # latitude
            longitude: 4.8374263     # longitude
          },
          distance: {
            max: 1000           # distance in meters
          }
        },
        path: ["headquartersGeoLocation"]  # property needs to be a geoLocation data type.
      }) {
        name
        headquartersGeoLocation {
          latitude
          longitude
        }
      }
    }
  }
"""

query_result = client.query.raw(get_publications_where)
print(query_result)

In [ ]:
from weaviate.classes.query import Filter
from weaviate.classes.query import GeoCoordinate

response = publications.query.fetch_objects(
    filters=(
        Filter
        .by_property("headquartersGeoLocation")
        .within_geo_range(
            coordinate=GeoCoordinate(
                latitude=52.39,
                longitude=4.84
            ),
            distance=1000  # In meters
        )
    )
)

for o in response.objects:
    print(o.properties)  # Inspect returned objects

## 按DATE数据类型
要按DATE数据类型属性进行过滤，请将日期/时间指定为RFC 3339时间戳，或客户端库兼容类型（例如 Pythondatetime对象）。

In [ ]:
from datetime import datetime, timezone
from weaviate.classes.query import Filter, MetadataQuery

# Set the timezone for avoidance of doubt
filter_time = datetime(2022, 6, 10).replace(tzinfo=timezone.utc)
# The filter threshold could also be an RFC 3339 timestamp, e.g.:
# filter_time = "2022-06-10T00:00:00.00Z"

response = collection.query.fetch_objects(
    limit=3,
    # This property (`some_date`) is a `DATE` datatype
    filters=Filter.by_property("some_date").greater_than(filter_time),
)

for o in response.objects:
    print(o.properties)  # Inspect returned objects

## 按元数据过滤
过滤器还可以处理元数据属性，例如对象 ID、属性长度和时间戳。

有关完整列表，请参阅API 参考：[过滤器](https://weaviate.io/developers/weaviate/api/graphql/filters#special-cases)。

### 按对象id



In [ ]:
target_id = "00037775-1432-35e5-bc59-443baaef7d80"
response = (
    client.query
    .get("Article", ["title"])
    .with_where({
        "path": "id",
        "operator": "Equal",
        "valueText": target_id
    })
    .with_additional("id")
    .do()
)

print(response)

In [ ]:
from weaviate.classes.query import Filter

collection = client.collections.get("Article")

target_id = "00037775-1432-35e5-bc59-443baaef7d80"
response = collection.query.fetch_objects(
    filters=Filter.by_id().equal(target_id)
)

for o in response.objects:
    print(o.properties)  # Inspect returned objects
    print(o.uuid)

### 按对象时间戳
此过滤器需要对属性时间戳进行索引。

In [ ]:
timestamp_str = "2020-01-01T00:00:00+00:00"

response = (
    client.query
    .get("Article", ["title"])
    .with_where({
        "path": ["_creationTimeUnix"],
        "operator": "GreaterThan",
        "valueDate": timestamp_str  # Can use either `valueDate` with a `RFC3339` datetime or `valueText` as Unix epoch milliseconds
    })
    .with_additional("creationTimeUnix")
    .with_limit(3)
    .do()
)

print(response)

In [ ]:
from datetime import datetime, timezone
from weaviate.classes.query import Filter, MetadataQuery

collection = client.collections.get("Article")

# Set the timezone for avoidance of doubt (otherwise the client will emit a warning)
filter_time = datetime(2020, 1, 1).replace(tzinfo=timezone.utc)

response = collection.query.fetch_objects(
    limit=3,
    filters=Filter.by_creation_time().greater_than(filter_time),
    return_metadata=MetadataQuery(creation_time=True)
)

for o in response.objects:
    print(o.properties)  # Inspect returned objects
    print(o.metadata.creation_time)  # Inspect object creation time

### 按对象属性长度
此过滤器需要对属性长度进行索引。

In [ ]:
response = (
    client.query
    .get("JeopardyQuestion", ["answer"])
    .with_where({
        "path": ["len(answer)"],
        "operator": "GreaterThan",
        "valueInt": 20
    })
    .with_limit(3)
    .do()
)

print(response)

In [ ]:
from weaviate.classes.query import Filter

collection = client.collections.get("JeopardyQuestion")

response = collection.query.fetch_objects(
    limit=3,
    filters=Filter.by_property("answer", length=True).greater_than(length_threshold),
)

for o in response.objects:
    print(o.properties)  # Inspect returned objects
    print(len(o.properties["answer"]))  # Inspect property length

## 按对象 null 状态
此过滤器要求对属性空状态进行索引。

In [ ]:
response = (
    client.query
    .get("JeopardyQuestion", ["points"])
    .with_where({
        "path": ["points"],
        "operator": "IsNull",
        "valueBoolean": True
    })
    .with_limit(3)
    .do()
)

print(response)

In [ ]:
from weaviate.classes.query import Filter

collection = client.collections.get("WineReview")

response = collection.query.fetch_objects(
    limit=3,
    # This requires the `country` property to be configured with `index_null_state=True``
    filters=Filter.by_property("country").is_none(True)  # Find objects where the `country` property is null
)

for o in response.objects:
    print(o.properties)  # Inspect returned objects

## 过滤器注意事项
Weaviate 将过滤条件转换为标记。默认标记方式为word。word标记器会保留字母数字字符，将其小写，并按空格拆分。它会将“Test_domain_weaviate”这样的字符串转换为“test”、“domain”和“weaviate”。

有关详细信息和其他标记化方法，[请参阅标记化](https://weaviate.io/developers/weaviate/config-refs/schema#tokenization)。

### 提高过滤性能
如果遇到过滤器性能缓慢的情况，请考虑添加limit参数或其他where运算符来限制数据集的大小。

### 过滤运算符列表

有关过滤器运算符的列表，请[参阅参考页](https://weaviate.io/developers/weaviate/api/graphql/filters#filter-structure)。